In [1]:
# =====================================================================
# MULTISOURCE PATCH EXTRACTION FOR DEEP LEARNING
# =====================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
import os
import time
import json
from pathlib import Path
from scipy.spatial import cKDTree
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from shapely.geometry import box, Point
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from rasterio.transform import rowcol
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# =====================================================================
# CONFIGURATION
# =====================================================================

ROOT_DIRECTORY = r'D:\Tese_data_refinada'
ROOT_DIRECTORY_DATA = os.path.join(ROOT_DIRECTORY, 'DATA')

# Tamanhos de patch a gerar.
# Para correr só um tamanho: PATCH_SIZES_TO_RUN = [7]
# Para testar vários tamanhos: PATCH_SIZES_TO_RUN = [3, 5, 7, 11, 15]
#PATCH_SIZES_TO_RUN = [3, 5, 7, 11, 15]
PATCH_SIZES_TO_RUN = [21]

PIXEL_SIZE = 25
BIOMASS_POOL = 'HEIGHT'

# Pastas raiz. Cada tamanho fica numa subpasta própria.
OUTPUT_ROOT_DIR = os.path.join(ROOT_DIRECTORY, '_multisource_patches_by_size')
VALIDATION_ROOT_DIR = os.path.join(ROOT_DIRECTORY, '_validation_qgis_by_size')
GEOTIFF_ROOT_DIR = os.path.join(ROOT_DIRECTORY, '_example_patches_by_size')

# O treino CNN atual lê GeoTIFFs. Por isso esta opção fica ligada.
EXPORT_GEOTIFFS_FOR_TRAINING = True

# Mantém também os tensores .npy, caso queiras usar mais tarde.
SAVE_TENSORS = True

# Validação QGIS com polígonos dos patches.
EXPORT_QGIS_VALIDATION = True

# Exportar apenas amostras válidas segundo o critério de NaNs.
# Recomendado: True.
EXPORT_ONLY_VALID_GEOTIFFS = True

# Paralelização.
N_JOBS = -1  # -1 usa todos os cores disponíveis

# Critério original: se um raster tiver mais de 50% de NaNs no patch,
# a amostra é marcada como inválida.
MAX_NAN_FRACTION_PER_RASTER = 0.50

# Validação básica dos tamanhos.
for ps in PATCH_SIZES_TO_RUN:
    if ps % 2 == 0:
        raise ValueError(f"PATCH_SIZE tem de ser ímpar para ter pixel central. Recebido: {ps}")
    if ps <= 0:
        raise ValueError(f"PATCH_SIZE tem de ser positivo. Recebido: {ps}")

os.makedirs(OUTPUT_ROOT_DIR, exist_ok=True)
os.makedirs(VALIDATION_ROOT_DIR, exist_ok=True)
os.makedirs(GEOTIFF_ROOT_DIR, exist_ok=True)


def get_dirs_for_patch_size(patch_size):
    """
    Devolve as pastas específicas para um determinado patch size.
    """
    tag = f'patch_{patch_size}x{patch_size}'

    output_dir = os.path.join(OUTPUT_ROOT_DIR, tag)
    validation_dir = os.path.join(VALIDATION_ROOT_DIR, tag)
    geotiff_dir = os.path.join(GEOTIFF_ROOT_DIR, tag)

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(validation_dir, exist_ok=True)
    os.makedirs(geotiff_dir, exist_ok=True)

    return output_dir, validation_dir, geotiff_dir


print(f"Root directory: {ROOT_DIRECTORY}")
print(f"Patch sizes to run: {PATCH_SIZES_TO_RUN}")
print(f"Pixel size: {PIXEL_SIZE} m")
print(f"Tensor output root: {OUTPUT_ROOT_DIR}")
print(f"GeoTIFF output root: {GEOTIFF_ROOT_DIR}")

print("\nEscala espacial:")
for ps in PATCH_SIZES_TO_RUN:
    width_m = ps * PIXEL_SIZE
    area_ha = (width_m * width_m) / 10000
    print(f"  {ps}x{ps}: {width_m} m x {width_m} m ≈ {area_ha:.2f} ha")


Root directory: D:\Tese_data_refinada
Patch sizes to run: [21]
Pixel size: 25 m
Tensor output root: D:\Tese_data_refinada\_multisource_patches_by_size
GeoTIFF output root: D:\Tese_data_refinada\_example_patches_by_size

Escala espacial:
  21x21: 525 m x 525 m ≈ 27.56 ha


In [3]:
# =====================================================================
# DISCOVER AND LOAD DATA
# =====================================================================

def discover_data_structure(root_dir):

    data_structure = {}
    
    for directory in os.listdir(root_dir):
        dir_path = os.path.join(root_dir, directory)
        
        if not os.path.isdir(dir_path):
            continue
        
        area_name = directory.split('_')[0]
        
        if area_name not in data_structure:
            data_structure[area_name] = {
                'gpkg': None,
                'rasters': [],
                'crs': None
            }
        
        # Find GeoPackage (reference points)
        for file in os.listdir(dir_path):
            if file.endswith('.gpkg'):
                gpkg_path = os.path.join(dir_path, file)
                data_structure[area_name]['gpkg'] = gpkg_path
                
                # Load and get CRS
                gdf = gpd.read_file(gpkg_path)
                data_structure[area_name]['crs'] = gdf.crs
                print(f"Found GeoPackage for {area_name}: {file}")
                print(f"  CRS: {gdf.crs}")
                print(f"  Points: {len(gdf)}")
            
            # Find rasters (satellite data)
            elif file.endswith('.tif'):
                raster_path = os.path.join(dir_path, file)
                data_structure[area_name]['rasters'].append(raster_path)
        
        if data_structure[area_name]['rasters']:
            print(f"  Found {len(data_structure[area_name]['rasters'])} rasters")
    
    return data_structure


# Discover data
print("Discovering data structure...")
data_structure = discover_data_structure(ROOT_DIRECTORY_DATA)

print(f"\nDiscovered {len(data_structure)} areas:")
for area, info in data_structure.items():
    print(f"  {area}:")
    print(f"    - GeoPackage: {os.path.basename(info['gpkg']) if info['gpkg'] else 'NOT FOUND'}")
    print(f"    - Rasters: {len(info['rasters'])}")
    print(f"    - CRS: {info['crs']}")

# Filter valid areas (those with both gpkg and rasters)
valid_areas = {k: v for k, v in data_structure.items() 
               if v['gpkg'] is not None and len(v['rasters']) > 0}

print(f"\nValid areas for processing: {list(valid_areas.keys())}")

Discovering data structure...
Found GeoPackage for Mafra: Mafra_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 463
  Found 306 rasters
Found GeoPackage for Monsanto: Monsanto_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 455
  Found 306 rasters
Found GeoPackage for Pombal: Pombal_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 850
  Found 306 rasters
Found GeoPackage for SerraLousa: SerraLousa_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 5825
  Found 306 rasters
Found GeoPackage for SintraCascais: SintraCascais_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 736
  Found 306 rasters
Found GeoPackage for VPA: VPA_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 535
  Found 306 rasters

Discovered 6 areas:
  Mafra:
    - GeoPackage: Mafra_20200000_GEDI_AGBD_ALL.gpkg
    - Rasters: 306
    - CRS: EPSG:3763
  Monsanto:
    - GeoPackage: Monsanto_20200000_GEDI_AGBD_ALL.gpkg
    - Rasters: 306
    - CRS: EPSG:3763
  Pombal:
    - GeoPackage: Pomb

In [4]:
# =====================================================================
# INTERPOLATION FUNCTIONS
# =====================================================================

def cubic_kernel(t):
    """
    Cubic interpolation kernel (Catmull-Rom spline).
    """
    t = np.abs(t)
    a = -0.5
    
    return np.where(
        t <= 1,
        (a + 2) * t**3 - (a + 3) * t**2 + 1,
        np.where(
            t < 2,
            a * t**3 - 5 * a * t**2 + 8 * a * t - 4 * a,
            0
        )
    )


def bicubic_interpolation(image, x, y, no_data_value):
    """
    Bicubic interpolation handling NaN values.
    
    Returns: interpolated_value (float), success (bool)
    """
    height, width = image.shape
    
    # Handle border cases
    if x < 1 or x > width - 3 or y < 1 or y > height - 3:
        return no_data_value, False
    
    # Floor values
    x0 = int(np.floor(x))
    y0 = int(np.floor(y))
    
    # Compute distances
    dx = x - x0
    dy = y - y0
    
    # Extract 4x4 neighborhood
    neighborhood = image[y0-1:y0+3, x0-1:x0+3].astype(np.float32)
    
    # Mask valid (non-nodata) values
    valid_mask = ~np.isin(neighborhood, no_data_value)
    
    if valid_mask.sum() < 8:  # Not enough valid neighbors
        return no_data_value, False
    
    # Replace nodata with mean of valid values
    valid_vals = neighborhood[valid_mask]
    if len(valid_vals) > 0:
        neighborhood[neighborhood == no_data_value] = np.mean(valid_vals)
    
    # Compute weights
    wx = cubic_kernel(np.arange(-1, 3) - dx)
    wy = cubic_kernel(np.arange(-1, 3) - dy)
    
    # Apply valid mask to weights
    wx_masked = wx * valid_mask.sum(axis=0)
    wy_masked = wy * valid_mask.sum(axis=1)
    
    # Normalize weights
    if np.sum(wx_masked) > 0:
        wx_masked /= np.sum(wx_masked)
    if np.sum(wy_masked) > 0:
        wy_masked /= np.sum(wy_masked)
    
    # Interpolate
    intermediate = np.dot(neighborhood, wx_masked)
    result = np.dot(intermediate, wy_masked)
    
    # Verify result is within valid range
    valid_range_min = np.min(valid_vals)
    valid_range_max = np.max(valid_vals)
    
    if not (valid_range_min <= result <= valid_range_max):
        return no_data_value, False
    
    return result, True


def bilinear_interpolation(image, x, y, no_data_value):
    """
    Bilinear interpolation handling NaN values.
    
    Returns: interpolated_value (float), success (bool)
    """
    height, width = image.shape
    
    # Get integer grid points
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    x1 = min(x0 + 1, width - 1)
    y1 = min(y0 + 1, height - 1)
    
    # Extract 2x2 neighborhood
    neighborhood = np.array([
        [image[y0, x0], image[y0, x1]],
        [image[y1, x0], image[y1, x1]]
    ], dtype=np.float32)
    
    # Valid mask
    valid_mask = ~np.isin(neighborhood, no_data_value)
    
    if valid_mask.sum() < 2:
        return no_data_value, False
    
    # Replace nodata with mean
    valid_vals = neighborhood[valid_mask]
    if len(valid_vals) > 0:
        neighborhood[~valid_mask] = np.mean(valid_vals)
    
    # Compute weights
    wx = x - x0
    wy = y - y0
    
    # Interpolate along x
    weights_x = np.array([1 - wx, wx])
    sum_weights_x = np.dot(weights_x, valid_mask)
    
    fx_y = np.dot(weights_x, neighborhood)
    fx_y = np.where(sum_weights_x > 0, fx_y / sum_weights_x, np.nan)
    
    # Interpolate along y
    weights_y = np.array([1 - wy, wy])
    valid_y = ~np.isnan(fx_y)
    
    if not valid_y.any():
        return no_data_value, False
    
    f_xy = np.dot(weights_y, np.nan_to_num(fx_y))
    f_xy /= np.dot(weights_y, valid_y.astype(float))
    
    # Verify range
    if not (np.min(valid_vals) <= f_xy <= np.max(valid_vals)):
        return no_data_value, False
    
    return f_xy, True


def get_raster_value_at_coord(raster_data, no_data, transform, height, width, coord_x, coord_y, use_bicubic=True):
    """
    Extract interpolated value from raster at geographic coordinate.
    
    Returns: (value, interpolation_method_used)
    """
    
    # CORREÇÃO: Obter coordenadas em float (com casas decimais) para permitir interpolação
    col, row = ~transform * (coord_x, coord_y)
    
    # Para ler o array e validar limites, usamos os valores inteiros (Nearest)
    int_row, int_col = int(np.floor(row)), int(np.floor(col))
        
    # Check bounds
    if int_row < 0 or int_col < 0 or int_row >= height or int_col >= width:
        return np.nan, 'out_of_bounds'
        
    # Try to read nearest neighbor value
    nn_value = raster_data[int_row, int_col]
    
    if nn_value == no_data or np.isnan(nn_value):
        return np.nan, 'nodata'
    
    # Try bicubic (NOTA: passamos 'col' e 'row' que têm as casas decimais!)
    if use_bicubic and int_row > 0 and int_col > 0 and \
       int_row < height - 2 and int_col < width - 2:
        value, success = bicubic_interpolation(raster_data, col, row, no_data)
        if success and not np.isnan(value):
            return value, 'bicubic'
    
    # Try bilinear
    if int_row < height - 1 and int_col < width - 1:
        value, success = bilinear_interpolation(raster_data, col, row, no_data)
        if success and not np.isnan(value):
            return value, 'bilinear'
    
    # Return nearest neighbor fallback
    return nn_value, 'nearest'


In [5]:
# =====================================================================
# PATCH GENERATION UTILITIES
# =====================================================================

def generate_patch_coordinates(center_x, center_y, patch_size=15, pixel_size=25):
    """
    Generate geographic coordinates for a patch grid.

    The patch center corresponds exactly to (center_x, center_y).
    patch_size must be odd, so that there is a true central pixel.

    The coordinates generated here are pixel CENTERS.
    Example with patch_size=15 and pixel_size=25:
      - 15 centers spaced by 25 m
      - full patch footprint = 15 * 25 = 375 m
    """
    if patch_size % 2 == 0:
        raise ValueError("patch_size must be odd to have a central pixel.")

    half_patch = (patch_size - 1) / 2.0
    offset = half_patch * pixel_size

    coords = np.zeros((patch_size, patch_size, 2), dtype=np.float64)

    for i in range(patch_size):
        for j in range(patch_size):
            # i is row (y direction), j is column (x direction)
            x = center_x - offset + j * pixel_size
            y = center_y + offset - i * pixel_size  # top is +y
            coords[i, j, 0] = x
            coords[i, j, 1] = y

    return coords


def get_patch_bounds(center_x, center_y, patch_size=15, pixel_size=25):
    """
    Get full patch footprint bounds (minx, miny, maxx, maxy).

    This uses pixel EDGES, not only the outer pixel centers.
    Therefore:
      3x3  with 25 m pixels -> 75 m footprint
      15x15 with 25 m pixels -> 375 m footprint
    """
    if patch_size % 2 == 0:
        raise ValueError("patch_size must be odd to have a central pixel.")

    half_width = (patch_size * pixel_size) / 2.0

    minx = center_x - half_width
    maxx = center_x + half_width
    miny = center_y - half_width
    maxy = center_y + half_width

    return (minx, miny, maxx, maxy)


def create_patch_polygon(center_x, center_y, patch_size=15, pixel_size=25):
    """
    Create Shapely polygon for full patch extent (for QGIS).
    """
    minx, miny, maxx, maxy = get_patch_bounds(center_x, center_y, patch_size, pixel_size)
    return box(minx, miny, maxx, maxy)

print("Patch utilities loaded successfully!")


Patch utilities loaded successfully!


In [6]:
# =====================================================================
# PROCESSING FUNCTIONS
# =====================================================================

import time
import gc
from joblib import Parallel, delayed


def process_single_raster(c_idx, r_path, r_name, N, patch_size, precomputed_coords, no_data_val=None):
    """
    Lê um único raster e extrai os patches para todos os pontos.
    Corre em paralelo para cada raster.
    """
    X_raster = np.zeros((N, patch_size, patch_size), dtype=np.float32)
    nan_counts = np.zeros(N, dtype=int)

    meta_raster = {
        n_idx: {'bicubic': 0, 'bilinear': 0, 'nearest': 0,
                'nodata': 0, 'out_of_bounds': 0, 'total': patch_size**2}
        for n_idx in range(N)
    }

    with rasterio.open(r_path) as src:
        raster_data = src.read(1)
        no_data = src.nodata if src.nodata is not None else no_data_val
        transform = src.transform
        h, w = src.height, src.width

        for n_idx in range(N):
            patch_coords = precomputed_coords[n_idx]
            patch = np.zeros((patch_size, patch_size), dtype=np.float32)
            nan_cnt = 0

            for i in range(patch_size):
                for j in range(patch_size):
                    px, py = patch_coords[i, j]
                    val, method = get_raster_value_at_coord(
                        raster_data, no_data, transform, h, w, px, py
                    )
                    patch[i, j] = val
                    meta_raster[n_idx][method] += 1

                    if np.isnan(val):
                        nan_cnt += 1

            X_raster[n_idx, :, :] = patch
            nan_counts[n_idx] = nan_cnt

    return c_idx, r_name, X_raster, nan_counts, meta_raster


def build_metadata_dict(result):
    """
    Cria o metadata.json compatível com os notebooks de treino.
    """
    X = result['X']

    return {
        'patch_size': int(result['patch_size']),
        'pixel_size': float(result['pixel_size']),
        'num_channels': int(len(result['raster_names'])),
        'num_samples': int(X.shape[0]),
        'raster_names': result['raster_names'],
        'valid_sample_indices': [int(v) for v in result['valid_indices']],
        'crs': str(result['crs']),
        'nan_percentage': float(np.isnan(X).sum() / X.size * 100),
        'patch_width_m': float(result['patch_size'] * result['pixel_size']),
        'patch_area_ha': float(((result['patch_size'] * result['pixel_size']) ** 2) / 10000)
    }


def save_results_to_disk(all_results, output_base_dir):
    """
    Guarda X.npy, y.npy, point_ids.npy e metadata.json por área.
    """
    print("\nSaving tensors to disk...")

    for area_name, result in all_results.items():
        area_output_dir = os.path.join(output_base_dir, area_name)
        os.makedirs(area_output_dir, exist_ok=True)

        np.save(os.path.join(area_output_dir, 'X.npy'), result['X'])
        np.save(os.path.join(area_output_dir, 'y.npy'), result['y'])
        np.save(os.path.join(area_output_dir, 'point_ids.npy'), np.array(result['valid_indices'], dtype=np.int64))

        metadata = build_metadata_dict(result)

        with open(os.path.join(area_output_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2)

        print(f"\n{area_name}:")
        print(f"  X: {result['X'].shape} ({result['X'].nbytes / 1e9:.2f} GB)")
        print(f"  y: {result['y'].shape} ({result['y'].nbytes / 1e6:.2f} MB)")
        print(f"  Saved to: {area_output_dir}")

    print("\n✓ Tensors saved successfully!")


def export_qgis_validation_outputs(all_results, validation_dir):
    """
    Guarda os polígonos dos patches para validação visual em QGIS.
    """
    print("\nGenerating QGIS validation GeoPackages...")

    for area_name, result in all_results.items():
        samples = result['samples']

        patch_geometries = []
        patch_attributes = {
            'point_id': [],
            'target': [],
            'valid': [],
            'center_x': [],
            'center_y': [],
            'nan_count': [],
            'nan_percent': []
        }

        for sample in samples:
            point_id = sample['point_id']
            center_x, center_y = sample['center']

            patch_poly = create_patch_polygon(
                center_x, center_y,
                result['patch_size'], result['pixel_size']
            )
            patch_geometries.append(patch_poly)

            patch_attributes['point_id'].append(point_id)
            patch_attributes['target'].append(float(sample['target']))
            patch_attributes['valid'].append(bool(sample['valid']))
            patch_attributes['center_x'].append(float(center_x))
            patch_attributes['center_y'].append(float(center_y))
            patch_attributes['nan_count'].append(int(sample['nan_count_total']))
            patch_attributes['nan_percent'].append(float(sample['nan_percent_total']))

        gdf_patches = gpd.GeoDataFrame(
            patch_attributes,
            geometry=patch_geometries,
            crs=result['crs']
        )

        output_path = os.path.join(
            validation_dir,
            f'{area_name}_patch_extents.gpkg'
        )
        gdf_patches.to_file(output_path, driver='GPKG')
        print(f"  {area_name}: saved {output_path}")

    print("\n✓ QGIS validation files saved!")


def export_multiband_patch_geotiff(patch_stack, raster_names, bounds, output_path, crs):
    """
    Exporta um patch multisensor como GeoTIFF multibanda georreferenciado.

    patch_stack shape: (C, H, W)
    """
    minx, miny, maxx, maxy = bounds

    count, height, width = patch_stack.shape
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=count,
        dtype=rasterio.float32,
        crs=crs,
        transform=transform,
        nodata=np.nan
    ) as dst:
        for band_idx, raster_name in enumerate(raster_names, start=1):
            dst.write(patch_stack[band_idx - 1].astype(np.float32), band_idx)
            dst.set_band_description(band_idx, raster_name)


def export_training_geotiffs(all_results, geotiff_base_dir, only_valid=True):
    """
    Exporta GeoTIFFs multibanda compatíveis com os notebooks de treino CNN.

    Estrutura criada:
      _example_patches_by_size/
        patch_7x7/
          Mafra/
            metadata.json
            point_0000_multisensor_patch.tif
            point_0001_multisensor_patch.tif
            ...
    """
    print("\nExporting training GeoTIFF patches...")

    for area_name, result in all_results.items():
        area_geotiff_dir = os.path.join(geotiff_base_dir, area_name)
        os.makedirs(area_geotiff_dir, exist_ok=True)

        # Guardar metadata junto aos GeoTIFFs, porque os notebooks de treino procuram
        # o metadata.json dentro da mesma pasta dos .tif.
        metadata = build_metadata_dict(result)
        with open(os.path.join(area_geotiff_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2)

        # Guardar também y e point_ids por conveniência. O treino atual não precisa,
        # porque cruza os point_id com o GeoPackage, mas estes ficheiros ajudam a auditar.
        np.save(os.path.join(area_geotiff_dir, 'y.npy'), result['y'])
        np.save(os.path.join(area_geotiff_dir, 'point_ids.npy'), np.array(result['valid_indices'], dtype=np.int64))

        X = result['X']
        raster_names = result['raster_names']
        valid_indices = result['valid_indices']
        gdf_points = result['gdf_points']
        crs = result['crs']
        patch_size = result['patch_size']
        pixel_size = result['pixel_size']

        exported = 0

        # X já contém apenas amostras válidas. valid_indices mapeia cada linha de X
        # para o índice original no GeoPackage.
        for x_row, original_idx in enumerate(valid_indices):
            original_idx = int(original_idx)

            center_x = float(gdf_points.iloc[original_idx].geometry.x)
            center_y = float(gdf_points.iloc[original_idx].geometry.y)

            bounds = get_patch_bounds(center_x, center_y, patch_size, pixel_size)

            output_file = os.path.join(
                area_geotiff_dir,
                f'point_{original_idx:04d}_multisensor_patch.tif'
            )

            export_multiband_patch_geotiff(
                X[x_row],
                raster_names,
                bounds,
                output_file,
                crs
            )

            exported += 1

            if exported % 500 == 0:
                print(f"  {area_name}: {exported} GeoTIFFs exported...")

        print(f"  {area_name}: exported {exported} GeoTIFFs to {area_geotiff_dir}")

    print("\n✓ Training GeoTIFFs exported!")


def print_validation_report(all_results):
    """
    Relatório rápido de validação após extração.
    """
    print("\n" + "="*70)
    print("MULTISOURCE PATCH EXTRACTION - VALIDATION REPORT")
    print("="*70)

    for area_name, result in all_results.items():
        X = result['X']
        y = result['y']
        samples = result['samples']
        raster_names = result['raster_names']

        print(f"\nArea: {area_name}")
        print("-" * 70)

        total_samples = len(samples)
        valid_samples = sum(1 for s in samples if s['valid'])
        invalid_samples = total_samples - valid_samples

        print(f"Patch size: {result['patch_size']}x{result['patch_size']}")
        print(f"Patch width: {result['patch_size'] * result['pixel_size']} m")
        print(f"Patch area: {((result['patch_size'] * result['pixel_size']) ** 2) / 10000:.2f} ha")
        print(f"Total reference points: {total_samples}")
        print(f"Valid patch samples: {valid_samples}")
        print(f"Invalid patch samples: {invalid_samples} ({100*invalid_samples/total_samples:.1f}%)")

        print(f"\nTensor information:")
        print(f"  Shape: {X.shape} (N, C, H, W)")
        print(f"  Size: {X.nbytes / 1e9:.3f} GB")
        print(f"  Data type: {X.dtype}")

        print(f"\nChannels ({len(raster_names)} total):")
        for i, name in enumerate(raster_names[:10]):
            print(f"  {i}: {name}")
        if len(raster_names) > 10:
            print(f"  ... and {len(raster_names) - 10} more")

        nan_count = np.isnan(X).sum()
        total_vals = X.size
        nan_pct = 100 * nan_count / total_vals

        print(f"\nNaN statistics:")
        print(f"  Total NaN values: {nan_count}/{total_vals} ({nan_pct:.4f}%)")

        valid_targets = y[~np.isnan(y)]
        if len(valid_targets) > 0:
            print(f"\nTarget statistics:")
            print(f"  Mean: {np.mean(valid_targets):.2f}")
            print(f"  Std:  {np.std(valid_targets):.2f}")
            print(f"  Min:  {np.min(valid_targets):.2f}")
            print(f"  Max:  {np.max(valid_targets):.2f}")

    print("\n✓ Validation report complete!")


def extract_patches_for_size(patch_size):
    """
    Extrai patches para todas as áreas válidas com um dado patch_size.
    """
    print("\n" + "="*70)
    print(f"STARTING PATCH EXTRACTION: {patch_size}x{patch_size}")
    print("="*70)

    all_results = {}

    for area_name, area_info in valid_areas.items():
        print(f"\nProcessing area: {area_name}")
        area_start_time = time.perf_counter()

        gdf_points = gpd.read_file(area_info['gpkg'])
        raster_paths = sorted(area_info['rasters'])

        N = len(gdf_points)
        C = len(raster_paths)
        raster_names = [Path(p).stem for p in raster_paths]

        print("  Pre-computing patch coordinates...")
        precomputed_coords = [
            generate_patch_coordinates(row.geometry.x, row.geometry.y, patch_size, PIXEL_SIZE)
            for _, row in gdf_points.iterrows()
        ]

        X = np.zeros((N, C, patch_size, patch_size), dtype=np.float32)
        y = (
            gdf_points[BIOMASS_POOL].values.astype(np.float32)
            if BIOMASS_POOL in gdf_points.columns
            else np.zeros(N, dtype=np.float32)
        )

        is_valid_sample = np.ones(N, dtype=bool)
        sample_metadata = {n_idx: {} for n_idx in range(N)}
        nan_total_per_sample = np.zeros(N, dtype=np.int64)

        print(f"  Extracting patches from {C} rasters using Parallel Processing...")
        results = Parallel(n_jobs=N_JOBS, backend="loky")(
            delayed(process_single_raster)(
                c_idx, r_path, raster_names[c_idx], N, patch_size, precomputed_coords
            )
            for c_idx, r_path in enumerate(raster_paths)
        )

        for res in results:
            c_idx, r_name, X_raster, nan_counts, meta_raster = res

            X[:, c_idx, :, :] = X_raster
            nan_total_per_sample += nan_counts

            for n_idx in range(N):
                sample_metadata[n_idx][r_name] = meta_raster[n_idx]

                if nan_counts[n_idx] / (patch_size**2) > MAX_NAN_FRACTION_PER_RASTER:
                    is_valid_sample[n_idx] = False

        valid_indices = np.where(is_valid_sample)[0]
        X_final = X[valid_indices]
        y_final = y[valid_indices]

        formatted_samples = []
        for n_idx in range(N):
            total_values_sample = C * patch_size * patch_size
            nan_pct_sample = 100 * nan_total_per_sample[n_idx] / total_values_sample

            formatted_samples.append({
                'point_id': int(n_idx),
                'center': (
                    float(gdf_points.iloc[n_idx].geometry.x),
                    float(gdf_points.iloc[n_idx].geometry.y)
                ),
                'target': float(y[n_idx]),
                'valid': bool(is_valid_sample[n_idx]),
                'raster_names': raster_names,
                'metadata': sample_metadata[n_idx],
                'nan_count_total': int(nan_total_per_sample[n_idx]),
                'nan_percent_total': float(nan_pct_sample)
            })

        all_results[area_name] = {
            'X': X_final,
            'y': y_final,
            'raster_names': raster_names,
            'valid_indices': valid_indices.tolist(),
            'samples': formatted_samples,
            'gdf_points': gdf_points,
            'patch_size': patch_size,
            'pixel_size': PIXEL_SIZE,
            'crs': gdf_points.crs
        }

        elapsed = time.perf_counter() - area_start_time
        print(f"Finished {area_name} in {elapsed/60:.2f} minutes.")
        print(f"  Valid samples: {len(valid_indices)}/{N}")
        print(f"  X_final shape: {X_final.shape}")

        # Libertar memória intermediária da área.
        del X
        gc.collect()

    print("\n" + "="*70)
    print(f"EXTRACTION COMPLETE: {patch_size}x{patch_size}")
    print("="*70)

    return all_results

print("Processing functions loaded successfully!")


Processing functions loaded successfully!


In [7]:
# =====================================================================
# RUN PATCH EXTRACTION FOR SELECTED PATCH SIZES
# =====================================================================

run_summaries = []

for patch_size in PATCH_SIZES_TO_RUN:
    output_base_dir, validation_dir, geotiff_dir = get_dirs_for_patch_size(patch_size)

    print("\n" + "#"*80)
    print(f"RUNNING PATCH SIZE {patch_size}x{patch_size}")
    print(f"Tensor output: {output_base_dir}")
    print(f"GeoTIFF output: {geotiff_dir}")
    print("#"*80)

    all_results = extract_patches_for_size(patch_size)

    if SAVE_TENSORS:
        save_results_to_disk(all_results, output_base_dir)

    if EXPORT_QGIS_VALIDATION:
        export_qgis_validation_outputs(all_results, validation_dir)

    if EXPORT_GEOTIFFS_FOR_TRAINING:
        export_training_geotiffs(
            all_results,
            geotiff_dir,
            only_valid=EXPORT_ONLY_VALID_GEOTIFFS
        )

    print_validation_report(all_results)

    # Guardar resumo leve, sem manter X em memória para todos os tamanhos.
    for area_name, result in all_results.items():
        X = result['X']
        run_summaries.append({
            'patch_size': patch_size,
            'area': area_name,
            'num_samples': int(X.shape[0]),
            'num_channels': int(X.shape[1]),
            'height': int(X.shape[2]),
            'width': int(X.shape[3]),
            'nan_percentage': float(np.isnan(X).sum() / X.size * 100),
            'tensor_output_dir': os.path.join(output_base_dir, area_name),
            'geotiff_output_dir': os.path.join(geotiff_dir, area_name)
        })

    # Libertar memória antes do próximo tamanho.
    del all_results
    gc.collect()

summary_df = pd.DataFrame(run_summaries)
display(summary_df)

summary_csv = os.path.join(OUTPUT_ROOT_DIR, 'patch_size_generation_summary.csv')
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved summary: {summary_csv}")

if EXPORT_GEOTIFFS_FOR_TRAINING:
    geotiff_summary_csv = os.path.join(GEOTIFF_ROOT_DIR, 'patch_size_generation_summary.csv')
    summary_df.to_csv(geotiff_summary_csv, index=False)
    print(f"Saved GeoTIFF summary: {geotiff_summary_csv}")



################################################################################
RUNNING PATCH SIZE 21x21
Tensor output: D:\Tese_data_refinada\_multisource_patches_by_size\patch_21x21
GeoTIFF output: D:\Tese_data_refinada\_example_patches_by_size\patch_21x21
################################################################################

STARTING PATCH EXTRACTION: 21x21

Processing area: Mafra
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished Mafra in 19.66 minutes.
  Valid samples: 376/463
  X_final shape: (376, 306, 21, 21)

Processing area: Monsanto
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished Monsanto in 18.93 minutes.
  Valid samples: 402/455
  X_final shape: (402, 306, 21, 21)

Processing area: Pombal
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished Pombal in 30.24 minutes.
  Valid samp

,patch_size,area,num_samples,num_channels,height,width,nan_percentage,tensor_output_dir,geotiff_output_dir
0,21,Mafra,376,306,21,21,0.173420,D:\Tese_data_refinada\_multisource_patches_by_...,D:\Tese_data_refinada\_example_patches_by_size...
1,21,Monsanto,402,306,21,21,0.205251,D:\Tese_data_refinada\_multisource_patches_by_...,D:\Tese_data_refinada\_example_patches_by_size...
2,21,Pombal,777,306,21,21,0.145404,D:\Tese_data_refinada\_multisource_patches_by_...,D:\Tese_data_refinada\_example_patches_by_size...
3,21,SerraLousa,5187,306,21,21,0.189543,D:\Tese_data_refinada\_multisource_patches_by_...,D:\Tese_data_refinada\_example_patches_by_size...
4,21,SintraCascais,555,306,21,21,0.207751,D:\Tese_data_refinada\_multisource_patches_by_...,D:\Tese_data_refinada\_example_patches_by_size...
5,21,VPA,503,306,21,21,0.103162,D:\Tese_data_refinada\_multisource_patches_by_...,D:\Tese_data_refinada\_example_patches_by_size...



Saved summary: D:\Tese_data_refinada\_multisource_patches_by_size\patch_size_generation_summary.csv
Saved GeoTIFF summary: D:\Tese_data_refinada\_example_patches_by_size\patch_size_generation_summary.csv


In [8]:
# =====================================================================
# LOAD AND VERIFY SAVED OUTPUTS
# =====================================================================

def load_area_tensors(area_name, patch_size, output_root_dir=OUTPUT_ROOT_DIR):
    """
    Load tensors and metadata for an area and patch size.
    """
    output_base_dir, _, _ = get_dirs_for_patch_size(patch_size)
    area_dir = os.path.join(output_base_dir, area_name)

    X = np.load(os.path.join(area_dir, 'X.npy'))
    y = np.load(os.path.join(area_dir, 'y.npy'))

    with open(os.path.join(area_dir, 'metadata.json'), 'r', encoding='utf-8') as f:
        metadata = json.load(f)

    return X, y, metadata


def verify_geotiff_area(area_name, patch_size):
    """
    Verifica rapidamente uma pasta de GeoTIFFs para treino.
    """
    _, _, geotiff_dir = get_dirs_for_patch_size(patch_size)
    area_dir = os.path.join(geotiff_dir, area_name)

    tif_files = sorted(Path(area_dir).glob("*.tif"))
    metadata_path = Path(area_dir) / "metadata.json"

    print(f"\nGeoTIFF verification")
    print(f"Area: {area_name}")
    print(f"Patch size: {patch_size}x{patch_size}")
    print(f"Folder: {area_dir}")
    print(f"Num GeoTIFFs: {len(tif_files)}")
    print(f"metadata.json exists? {metadata_path.exists()}")

    if tif_files:
        with rasterio.open(tif_files[0]) as src:
            print(f"First file: {tif_files[0].name}")
            print(f"  shape: ({src.count}, {src.height}, {src.width})")
            print(f"  CRS: {src.crs}")
            print(f"  pixel size: ({abs(src.transform.a):.3f}, {abs(src.transform.e):.3f})")
            print(f"  first band name: {src.descriptions[0]}")


# Exemplo de verificação rápida.
if len(PATCH_SIZES_TO_RUN) > 0 and len(valid_areas) > 0:
    example_patch_size = PATCH_SIZES_TO_RUN[0]
    first_area = list(valid_areas.keys())[0]

    if SAVE_TENSORS:
        print(f"Loading tensors for area={first_area}, patch_size={example_patch_size}...")
        X, y, metadata = load_area_tensors(first_area, example_patch_size)

        print("\nLoaded successfully!")
        print(f"  X shape: {X.shape}")
        print(f"  y shape: {y.shape}")
        print(f"  patch_size metadata: {metadata['patch_size']}")
        print(f"  patch_width_m: {metadata['patch_width_m']}")
        print(f"  patch_area_ha: {metadata['patch_area_ha']:.2f}")

    if EXPORT_GEOTIFFS_FOR_TRAINING:
        verify_geotiff_area(first_area, example_patch_size)


Loading tensors for area=Mafra, patch_size=21...



Loaded successfully!
  X shape: (376, 306, 21, 21)
  y shape: (376,)
  patch_size metadata: 21
  patch_width_m: 525.0
  patch_area_ha: 27.56

GeoTIFF verification
Area: Mafra
Patch size: 21x21
Folder: D:\Tese_data_refinada\_example_patches_by_size\patch_21x21\Mafra
Num GeoTIFFs: 376
metadata.json exists? True
First file: point_0006_multisensor_patch.tif
  shape: (306, 21, 21)
  CRS: EPSG:3763
  pixel size: (25.000, 25.000)
  first band name: Mafra_20200000_ALOS2_BETAHH_A
